# 🏆 [Day 34] 실전 GDS 커뮤니티 탐지, 유사도 & 최단 경로 핸즈온 워크북

> **학습 목표**:
> 단순 타이핑 노동이 아닌, **'왜 이렇게 설계했는가?(Thinking Point)'**와 **'비교 대조군 실험(What-If Simulation)'**을 통해 GDS 커뮤니티 분할, Jaccard 유사도, 가중치 최단 경로의 본질을 100% 체득합니다.
>
> 1. 🧐 **[생각하기 1]**: 왜 항공망에서 `UNDIRECTED`(무방향)와 복합 가중치(`hours`, `km`, `airlines`)를 투영했는가?
> 2. 👥 **[커뮤니티 탐지]**: `Leiden` vs `Louvain` vs `LPA` 모듈러리티 품질 및 군집 크기 분포 실측
> 3. 🎯 **[생각하기 2 & 유사도]**: 왜 `Jaccard` 유사도로 쌍둥이 대체 허브 공항을 추천할 수 있는가?
> 4. 🚀 **[가중치 최단 경로]**: 인천(ICN) ➔ 두바이(DXB) 최단 비행시간(hours) 항로 추적 (Dijkstra)
> 5. 🔬 **[비교 실험 1]**: 가중치를 뺐을 때(단순 Hop수)와 비행시간 가중치 경로가 어떻게 달라지는가?
> 6. 💥 **[비교 실험 2]**: 관계 가중치를 주면 모듈러리티(Modularity) 점수가 왜 오히려 떨어질까?

## 0. 환경 설정 및 Neo4j GDS 연결 (로컬 실습 전용 DB)

In [ ]:
# [제공 코드] Neo4j 연결: 교재 실습은 로컬 Neo4j (bolt://localhost:7689) 전용입니다.
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: 교재는 100% 로컬 Neo4j 인스턴스 (7689) 강제
env_file = os.path.join(os.path.abspath(''), '.env')
if os.path.exists(env_file):
    load_dotenv(env_file, override=True)

# 상위 클라우드 Aura 환경변수가 상속되었을 경우 로컬 7689로 강제 치환
raw_uri = os.getenv('NEO4J_URI', '')
if not raw_uri or 'databases.neo4j.io' in raw_uri:
    NEO4J_URI = 'bolt://localhost:7689'
    NEO4J_USER = 'neo4j'
    NEO4J_PASSWORD = 'test0011'
else:
    NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7689')
    NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
    NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'test0011')

# 2) 드라이버 연결
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()

# 3) 공용 Cypher 실행 헬퍼
def run_cypher(query, **params):
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

print('Neo4j 연결:', NEO4J_URI)


## 🧐 [Thinking Point 1] 투영 전 질문: "왜 UNDIRECTED와 복합 가중치를 함께 실을까?"
- **`UNDIRECTED`(무방향)**: 편도 노선 누락이나 방향성에 갇히지 않고, 공항 간 '상호 동일 생활권/교류권'을 완전한 무방향 네트워크로 묶어줍니다.
- **`hours` (비행시간)**: 대권 거리를 순항속도(800km/h)로 나눈 뒤 이착륙·환승 페널티 1시간을 가산하여 현실적인 최단 항로를 찾게 합니다.
- **`airlines` (운항사수)**: 운항 항공사가 많을수록 교류 밀도가 높은 '가까운 커뮤니티'로 간주하는 척도가 됩니다.

In [ ]:
# 기존 잔존 투영 정리 및 가중치 무방향 투영 생성
run_cypher("CALL gds.graph.drop('airMasterGraph', false) YIELD graphName")

proj_query = """
CALL gds.graph.project(
    'airMasterGraph',
    'Airport',
    {
        FLIGHT: {
            type: 'FLIGHT',
            orientation: 'UNDIRECTED',
            properties: ['km', 'hours', 'airlines']
        }
    }
)
YIELD graphName, nodeCount, relationshipCount, projectMillis
"""
proj_res = run_cypher(proj_query)[0]
print(f"⚡ 인메모리 투영 완료: {proj_res['graphName']}")
print(f"  • 노드 수: {proj_res['nodeCount']:,}개 | 엣지 수: {proj_res['relationshipCount']:,}건 (무방향 2배 검산)")
print(f"  • 투영 시간: {proj_res['projectMillis']} ms")

## 👥 [Part 1] 커뮤니티 탐지: Leiden vs Louvain
- 네트워크 내에서 서로 빽빽하게 연결된 '초국경 항공 권역(Hub Zone)'을 자동으로 찾아냅니다.

In [ ]:
# Leiden 알고리즘 실행 (GDS 2.5+)
comm_query = """
CALL gds.leiden.stream('airMasterGraph', {
    randomSeed: 42,
    concurrency: 1
})
YIELD nodeId, communityId
WITH gds.util.asNode(nodeId) AS n, communityId
RETURN communityId,
       count(n) AS airport_count,
       collect(n.country)[..3] AS sample_countries,
       collect(n.name)[..3] AS sample_airports
ORDER BY airport_count DESC
LIMIT 5
"""
df_comm = pd.DataFrame(run_cypher(comm_query))
print("📊 [상위 5대 항공 권역 커뮤니티 분포]")
display(df_comm)

## 🧐 [Thinking Point 2] "가중치(airlines)를 주면 모듈러리티가 왜 오히려 떨어질까?"
- 대형 허브 공항 사이에 수십 개의 항공사가 몰려 있어 가중치가 허브 링크로만 편중됩니다.
- 이로 인해 소형 공항들이 상대적으로 소외되어 전역 모듈러리티 점수는 무가중보다 낮게 나올 수 있습니다.
- **실험**: 무가중 vs 가중치 모듈러리티 점수를 직접 비교해 봅니다.

In [ ]:
# 모듈러리티 비교 통계 산출
stats_query = """
CALL gds.leiden.stats('airMasterGraph', { randomSeed: 42, concurrency: 1 })
YIELD modularity AS unweighted_modularity
RETURN round(unweighted_modularity, 4) AS unweighted_modularity
"""
unw_res = run_cypher(stats_query)[0]
print(f"  • 무가중치 Leiden 모듈러리티: {unw_res['unweighted_modularity']}")

## 🔗 [Part 2] 노드 유사도: Jaccard 유사도로 대체 가능한 쌍둥이 허브 찾기
- 연결된 이웃 공항(목적지)의 교집합과 합집합 비율을 계산하여, 가장 비슷한 노선망을 가진 '대체 허브'를 추천합니다.

In [ ]:
sim_query = """
CALL gds.nodeSimilarity.stream('airMasterGraph', {
    similarityCutoff: 0.35,
    topK: 3
})
YIELD node1, node2, similarity
WITH gds.util.asNode(node1) AS a1, gds.util.asNode(node2) AS a2, similarity
RETURN a1.name AS airport_1, a1.country AS country_1,
       a2.name AS airport_2, a2.country AS country_2,
       round(similarity, 4) AS jaccard_similarity
ORDER BY jaccard_similarity DESC
LIMIT 5
"""
df_sim = pd.DataFrame(run_cypher(sim_query))
print("📊 [Top 5 쌍둥이 유사 공항쌍]")
display(df_sim)

## 🚀 [Part 3] 최단 경로 탐색: 인천(ICN) ➔ 두바이(DXB) 가중치 비행시간 최소화
- 단순 환승 횟수가 아닌, 누적 비행시간(`hours`)을 최소화하는 최적 항로를 Dijkstra 알고리즘으로 추적합니다.

In [ ]:
dijk_query = """
MATCH (src:Airport), (tgt:Airport)
WHERE (src.iata = 'ICN' OR src.name CONTAINS 'Incheon')
  AND (tgt.iata = 'DXB' OR tgt.name CONTAINS 'Dubai')
WITH src, tgt LIMIT 1
CALL gds.shortestPath.dijkstra.stream('airMasterGraph', {
    sourceNode: src,
    targetNode: tgt,
    relationshipWeightProperty: 'hours'
})
YIELD totalCost, nodeIds
RETURN [nid IN nodeIds | coalesce(gds.util.asNode(nid).iata, gds.util.asNode(nid).name)] AS route_iata,
       [nid IN nodeIds | gds.util.asNode(nid).name] AS route_names,
       round(totalCost, 2) AS flight_hours
"""
dijk_res = run_cypher(dijk_query)
if dijk_res:
    print(f"✈️ 최적 항로: {' ➔ '.join(dijk_res[0]['route_iata'])}")
    print(f"⏱️ 총 비행시간: {dijk_res[0]['flight_hours']} 시간")
else:
    print("⚠️ 경로를 찾지 못했습니다.")

## 🔬 [What-If 비교 실험] "비행시간 가중치를 뺐을 때(단순 Hop수)와 경로는 어떻게 달라질까?"

In [ ]:
# 무가중치(단순 Hop수) 최단 경로 탐색
unw_dijk_query = """
MATCH (src:Airport), (tgt:Airport)
WHERE (src.iata = 'ICN' OR src.name CONTAINS 'Incheon')
  AND (tgt.iata = 'DXB' OR tgt.name CONTAINS 'Dubai')
WITH src, tgt LIMIT 1
CALL gds.shortestPath.dijkstra.stream('airMasterGraph', {
    sourceNode: src,
    targetNode: tgt
})
YIELD totalCost, nodeIds
RETURN [nid IN nodeIds | coalesce(gds.util.asNode(nid).iata, gds.util.asNode(nid).name)] AS route_hops,
       toInteger(totalCost) AS hop_count
"""
unw_res = run_cypher(unw_dijk_query)
if unw_res:
    print(f"🛫 단순 환승 최소 항로: {' ➔ '.join(unw_res[0]['route_hops'])} (환승 횟수: {unw_res[0]['hop_count']} Hops)")

## 🧹 5. 메모리 즉시 반환 (Clean Slate)

In [ ]:
run_cypher("CALL gds.graph.drop('airMasterGraph') YIELD graphName")
print("✅ GDS 인메모리 서브그래프 안전 반환 완료 (RAM 100% 회수)")